# Aula 03 · Vídeo 3 — RAG com LangGraph (versão enriquecida)

Este notebook mantém o *tema original* e **adiciona um exemplo mais realista** usando um **arquivo público** (Project Gutenberg) como fonte externa.

## O que mudou
- Mantemos um mini‑corpus local para demonstração rápida.
- **Novo**: baixamos um texto público real (Project Gutenberg) e incluímos no índice.
- Mostramos o fluxo em **LangGraph**: `retrieve` → `generate` (com *estado compartilhado* e *fontes*).

## 1) Setup rápido
- Se tiver **OPENAI_API_KEY**, usamos `OpenAIEmbeddings` e um modelo `ChatOpenAI`.
- Sem chave: caímos em `FakeEmbeddings` e respostas mockadas (útil para gravar a aula).

In [7]:
import os, re, requests, pathlib
from typing import List, Dict
from typing_extensions import TypedDict
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

USE_OPENAI = bool(OPENAI_API_KEY)
USE_OPENAI

True

## 2) LLM e Embeddings
Preferimos OpenAI quando disponível. Caso contrário, usamos `FakeEmbeddings` (somente para fluxo didático).

In [8]:
try:
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
except Exception:
    ChatOpenAI = None
    OpenAIEmbeddings = None

from langchain_core.embeddings import FakeEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

if USE_OPENAI and ChatOpenAI and OpenAIEmbeddings:
    llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0)
    embeddings = OpenAIEmbeddings()
else:
    llm = None  # responderemos com mock no nó generate
    embeddings = FakeEmbeddings(size=768)

embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7d0766307550>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7d07282ed0d0>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

## 3) Corpus local + **arquivo público real**
Baixaremos **Alice's Adventures in Wonderland** (Project Gutenberg, domínio público) e somaremos ao nosso índice.

In [9]:
DATA_DIR = pathlib.Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Mini-corpus local (mantido do exemplo simples)
local_corpus = [
    "LangGraph organiza fluxos de IA como grafos de execução (nós/arestas).",
    "RAG combina recuperação de contexto com geração; útil quando o LLM não sabe algo do domínio.",
    "Agentes especializados podem ser conectados em grafo para compor soluções complexas.",
]
(DATA_DIR / "local_corpus.txt").write_text("\n".join(local_corpus), encoding="utf-8")

# Fonte pública (Project Gutenberg): Alice in Wonderland
alice_url = "https://www.gutenberg.org/cache/epub/11/pg11.txt"
alice_path = DATA_DIR / "alice_in_wonderland.txt"

try:
    r = requests.get(alice_url, timeout=30)
    if r.ok and len(r.text) > 2000:
        alice_path.write_text(r.text, encoding="utf-8")
        print("Baixado:", alice_path)
    else:
        print("Falha ao baixar. Usando conteúdo placeholder.")
        alice_path.write_text("Alice in Wonderland (resumo curto).", encoding="utf-8")
except Exception as e:
    print("Erro de rede (ok para demo offline):", e)
    alice_path.write_text("Alice in Wonderland (resumo curto OFFLINE).", encoding="utf-8")

len(alice_path.read_text(encoding="utf-8"))

Baixado: data/alice_in_wonderland.txt


163950

## 4) Chunking e índice vetorial (FAISS)
Criamos documentos a partir dos arquivos e os adicionamos ao FAISS para recuperação semântica.

In [10]:
try:
    from langchain_core.documents import Document
except ImportError:
    from langchain.schema import Document

def to_docs(text: str, source: str, chunk_size=800, overlap=120) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap)
    return [Document(page_content=ch, metadata={"source": source}) for ch in splitter.split_text(text)]

docs: List[Document] = []

docs += to_docs((DATA_DIR/"local_corpus.txt").read_text(encoding="utf-8"), source="local_corpus.txt")
docs += to_docs((DATA_DIR/"alice_in_wonderland.txt").read_text(encoding="utf-8"), source="alice_in_wonderland.txt")

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
len(docs)

270

## 5) Estado e nós (LangGraph)
Fluxo: **retrieve → generate**. O estado carrega a consulta, os `docs` recuperados e a `resposta`.

In [11]:
class RAGState(TypedDict, total=False):
    query: str
    docs: List[Dict]
    resposta: str
    fontes: List[str]

def retrieve_node(s: RAGState) -> RAGState:
    query = s.get("query", "")
    hits = retriever.invoke(query)
    # Guardamos apenas texto e metadados necessários
    docs_slim = [{"text": d.page_content, "source": d.metadata.get("source", "")} for d in hits]
    return {"docs": docs_slim, "fontes": list({d["source"] for d in docs_slim})}

def build_prompt(query: str, docs: List[Dict]) -> str:
    contexto = "\n\n".join([f"Fonte: {d['source']}\nTrecho:\n{d['text']}" for d in docs])
    instrucao = (
        "Você é um assistente que responde de forma objetiva. Use APENAS o contexto abaixo.\n"
        "Se não houver contexto suficiente, diga que não há informação.\n\n"
        f"Pergunta: {query}\n\nContexto:\n{contexto}\n\nResposta concisa:" 
    )
    return instrucao

def generate_node(s: RAGState) -> RAGState:
    query = s.get("query", "")
    docs = s.get("docs", [])
    if llm is None:
        # Sem LLM: devolve "resposta" textual simples com fontes
        if not docs:
            return {"resposta": "(offline) Não há contexto suficiente.", "fontes": []}
        snippet = docs[0]["text"][:240].replace("\n", " ")
        return {"resposta": f"(offline) Baseado no contexto: {snippet}...", "fontes": list({d['source'] for d in docs})}
    else:
        from langchain_core.prompts import PromptTemplate
        prompt = PromptTemplate.from_template("{instrucao}")
        chain = prompt | llm
        instrucao = build_prompt(query, docs)
        out = chain.invoke({"instrucao": instrucao}).content
        return {"resposta": out, "fontes": list({d['source'] for d in docs})}


## 6) Montagem do grafo e execução
Conectamos `retrieve` → `generate` e rodamos com diferentes perguntas para evidenciar o uso da fonte pública.

In [12]:
g = StateGraph(RAGState)
g.add_node("retrieve", retrieve_node)
g.add_node("generate", generate_node)
g.set_entry_point("retrieve")
g.add_edge("retrieve", "generate")
g.add_edge("generate", END)
app = g.compile()
print(app.get_graph().draw_ascii())

tests = [
    "Qual é a ideia de RAG e quando usar?",
    "Quem é Alice e o que acontece no País das Maravilhas?",
]

for q in tests:
    out = app.invoke({"query": q})
    print("\n=== Pergunta ===\n", q)
    print("\nResposta:\n", out.get("resposta", ""))
    print("\nFontes:", out.get("fontes", []))

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+----------+   
| retrieve |   
+----------+   
      *        
      *        
      *        
+----------+   
| generate |   
+----------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   

=== Pergunta ===
 Qual é a ideia de RAG e quando usar?

Resposta:
 RAG combina recuperação de contexto com geração e é útil quando o LLM não sabe algo do domínio.

Fontes: ['local_corpus.txt', 'alice_in_wonderland.txt']

=== Pergunta ===
 Quem é Alice e o que acontece no País das Maravilhas?

Resposta:
 Alice é a protagonista de "Alice no País das Maravilhas", um livro de Lewis Carroll. No início da história, ela se sente entediada e acaba seguindo um Coelho Branco que a leva a um mundo fantástico, onde vive diversas aventuras e encontra personagens peculiares. No final, ela acorda de um longo sono, percebendo que tudo foi um sonho.

Fontes: ['alice_in_wo

## 7) Observações didáticas
- O **retrieve** adiciona `docs` e `fontes` ao estado.
- O **generate** lê do estado e produz `resposta` (com LLM ou modo offline).
- Com **OpenAI** ativo, a resposta usa apenas o **contexto recuperado**.